In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Helper Functions

In [ ]:
def generate_checkerboard_points(board_size, square_size):
    objp = np.zeros((board_size[0] * board_size[1], 3), np.float32)
    objp[:, :2] = np.mgrid[0:board_size[0], 0:board_size[1]].T.reshape(-1, 2)
    objnew=objp[::-1]
    return objnew * square_size

def showExtrinsics(Checker_Points, rvecs, tvecs):
    """
    Visualizes all checkerboard poses in 3D using the extrinsic parameters
    returned from cv2.calibrateCamera.

    Parameters:
    - Checker_Points: np.ndarray of shape (N, 3), 3D checkerboard points in object coordinates
    - rvecs: list of rotation vectors (from calibrateCamera), one per view
    - tvecs: list of translation vectors (from calibrateCamera), one per view
    """
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    for rvec, tvec in zip(rvecs, tvecs):
        # Convert rotation vector to rotation matrix
        R, _ = cv2.Rodrigues(rvec)
        T = np.asarray(tvec).reshape(3, 1)

        # Transform the object points to the camera coordinate system
        transformed_points = (R @ Checker_Points.T + T).T  # shape (N, 3)

        # Plot the transformed checkerboard
        ax.plot(transformed_points[:, 0], transformed_points[:, 1], transformed_points[:, 2], 'o-')

    ax.set_xlabel('X (camera)')
    ax.set_ylabel('Y (camera)')
    ax.set_zlabel('Z (camera)')
    ax.set_title('Checkerboard Poses in Camera Coordinate System')
    ax.view_init(elev=20, azim=-60)
    ax.set_box_aspect([1,1,1])
    plt.grid(True)
    plt.tight_layout()
    plt.show()

#  Corner detection, Camera Calibration 

In [ ]:
#ImageArray  #load calibration images into array #image array should be 1616x1240xNumberofImages
camera_width=ImageArray.shape[0]
camera_height=ImageArray.shape[1]
Number_of_Images=ImageArray.shape[2] 
board_size = (24,17)  #CheckerboardSize 
worldPoints = generate_checkerboard_points(board_size, squareSize)  #Set up general checkerboard points in 3D object space
objectPoints = [worldPoints.astype(np.float32)] * Number_of_Images  #need list of object points for each image



#detectCorners
corners,ret=cv2.findChessboardCornersSB(ImageArray, board_size, cv2.CALIB_CB_NORMALIZE_IMAGE,cv2.CALIB_CB_EXHAUSTIVE)


# Calibrate camera using OpenCV:

criteriaEnd = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30000, 0.0001)  # Termination criteria for estimateCameraParams
reprojectionError1, camera_matrix1, dist_coeffs1, rotation_vectors, translation_vectors = cv2.calibrateCamera(
    objectPoints, corners, (camera_width, camera_height), None, None,
    flags=cv2.CALIB_FIX_K4 + cv2.CALIB_FIX_K5 + cv2.CALIB_FIX_K6,criteria=criteriaEnd
)

# Plotting Extrinsics

In [ ]:

showExtrinsics(worldPoints, rotation_vectors, translation_vectors)

